# ds004752 V5.6 Tranche 2.3 Feature Matrix Leakage Audit Plan

Purpose: record a claim-closed leakage-audit plan before feature matrix materialization or comparator execution.

Integrity boundary: this notebook does not materialize feature values, audit runtime comparator logs, train models, run comparators, compute efficacy metrics, or open claims.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update repo

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)

## 3. Confirm Tranche 2.3 code is present

In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -8

from pathlib import Path

required_files = [
    Path('src/v56/feature_matrix_leakage_audit_plan.py'),
    Path('configs/v56/feature_matrix_leakage_audit_plan.json'),
    Path('tests/unit/test_v56_feature_matrix_leakage_audit_plan.py'),
    Path('bootstrap/run_v56_feature_matrix_leakage_plan.sh'),
    Path('docs/25_v56_feature_matrix_leakage_audit_plan_runbook_2026-05-04.md'),
]
missing = [str(p) for p in required_files if not p.exists()]
print({'missing_required_files': missing})
assert not missing, missing

cli_text = Path('src/cli.py').read_text(encoding='utf-8')
assert 'v56-feature-matrix-leakage-plan' in cli_text
assert 'run_v56_feature_matrix_leakage_audit_plan' in cli_text
print('V5.6 Tranche 2.3 leakage-audit plan code is present.')

## 4. Configure source-of-truth paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260424T100159866284Z'
SPLIT_LOCK_RUN = DRIVE_ROOT / 'artifacts/v56_split_registry_lock/latest.txt'
FEATURE_PROVENANCE_RUN = DRIVE_ROOT / 'artifacts/v56_feature_provenance_populated/latest.txt'
FEATURE_MATRIX_PLAN_RUN = DRIVE_ROOT / 'artifacts/v56_feature_matrix_plan/latest.txt'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/v56_feature_matrix_leakage_audit_plan'

for path in [GATE0_RUN, SPLIT_LOCK_RUN, FEATURE_PROVENANCE_RUN, FEATURE_MATRIX_PLAN_RUN]:
    print(path, path.exists())
    assert path.exists(), f'Missing required path: {path}'
print('OUTPUT_ROOT =', OUTPUT_ROOT)

## 5. Preflight upstream artifacts

In [ ]:
import json

split_dir = Path(SPLIT_LOCK_RUN.read_text().strip())
feature_dir = Path(FEATURE_PROVENANCE_RUN.read_text().strip())
feature_plan_dir = Path(FEATURE_MATRIX_PLAN_RUN.read_text().strip())
split_lock = json.loads((split_dir / 'v56_split_registry_lock.json').read_text())
feature = json.loads((feature_dir / 'v56_feature_provenance_populated.json').read_text())
feature_plan = json.loads((feature_plan_dir / 'v56_feature_matrix_plan.json').read_text())
feature_plan_validation = json.loads((feature_plan_dir / 'v56_feature_matrix_plan_validation.json').read_text())

preflight = {
    'split_status': split_lock.get('status'),
    'n_folds': len(split_lock.get('folds', [])),
    'test_time_inference': split_lock.get('test_time_inference'),
    'feature_status': feature.get('status'),
    'missing_sources': feature.get('missing_sources'),
    'feature_plan_status': feature_plan.get('status'),
    'feature_plan_validation_status': feature_plan_validation.get('status'),
    'feature_plan_claim_closed': feature_plan.get('claim_closed'),
}
print(json.dumps(preflight, indent=2))

assert preflight['split_status'] == 'locked_subject_level_split_registry', preflight
assert preflight['n_folds'] == 30, preflight
assert preflight['test_time_inference']['modality'] == 'scalp_eeg_only', preflight
assert preflight['test_time_inference']['allow_ieeg'] is False, preflight
assert preflight['test_time_inference']['allow_beamforming_bridge'] is False, preflight
assert preflight['feature_status'] == 'populated_source_hashes_and_split_links', preflight
assert preflight['missing_sources'] == [], preflight
assert preflight['feature_plan_status'] == 'planned_feature_matrix_contract_recorded', preflight
assert preflight['feature_plan_validation_status'] == 'v56_feature_matrix_plan_validation_passed', preflight
assert preflight['feature_plan_claim_closed'] is True, preflight
print('Preflight passed: Tranche 2.1/2.2 artifacts are ready.')

## 6. Run V5.6 Tranche 2.3 leakage-audit plan

In [ ]:
%cd /content/eeg-ds004752
import subprocess

cmd = [
    'bash',
    'bootstrap/run_v56_feature_matrix_leakage_plan.sh',
    str(GATE0_RUN),
    str(SPLIT_LOCK_RUN),
    str(FEATURE_PROVENANCE_RUN),
    str(FEATURE_MATRIX_PLAN_RUN),
    str(OUTPUT_ROOT),
]
print('$', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('returncode:', result.returncode)
print('--- stdout ---')
print(result.stdout)
print('--- stderr ---')
print(result.stderr)
assert result.returncode == 0, result.stderr or result.stdout
latest_after_run = OUTPUT_ROOT / 'latest.txt'
print('latest_after_run_exists:', latest_after_run.exists(), latest_after_run)
assert latest_after_run.exists(), latest_after_run

## 7. Load latest leakage-audit plan artifact

In [ ]:
latest = OUTPUT_ROOT / 'latest.txt'
if not latest.exists():
    print('Missing latest pointer:', latest)
    print('OUTPUT_ROOT exists:', OUTPUT_ROOT.exists())
    if OUTPUT_ROOT.exists():
        print('OUTPUT_ROOT children:', sorted(str(p) for p in OUTPUT_ROOT.iterdir())[:20])
    raise AssertionError(latest)
run_dir = Path(latest.read_text().strip())
assert run_dir.exists(), run_dir

plan = json.loads((run_dir / 'v56_feature_matrix_leakage_audit_plan.json').read_text())
summary = json.loads((run_dir / 'v56_feature_matrix_leakage_audit_plan_summary.json').read_text())
validation = json.loads((run_dir / 'v56_feature_matrix_leakage_audit_plan_validation.json').read_text())

compact = {
    'run_dir': str(run_dir),
    'status': plan.get('status'),
    'validation_status': validation.get('status'),
    'claim_closed': plan.get('claim_closed'),
    'claim_ready': plan.get('claim_ready'),
    'n_locked_folds': plan.get('n_locked_folds'),
    'n_planned_audit_checks': len(plan.get('planned_audit_checks', [])),
    'feature_matrix_materialized': plan.get('scientific_boundary', {}).get('feature_matrix_materialized'),
    'runtime_comparator_logs_audited': plan.get('scientific_boundary', {}).get('runtime_comparator_logs_audited'),
    'model_training_run': plan.get('scientific_boundary', {}).get('model_training_run'),
    'efficacy_metrics_computed': plan.get('scientific_boundary', {}).get('efficacy_metrics_computed'),
}
print(json.dumps(compact, indent=2))

## 8. Integrity assertions

In [ ]:
assert plan['status'] == 'planned_feature_matrix_leakage_audit_recorded', plan
assert validation['status'] == 'v56_feature_matrix_leakage_audit_plan_validation_passed', validation
assert validation['blocking_errors'] == [], validation
assert plan['claim_closed'] is True, plan
assert plan['claim_ready'] is False, plan
assert plan['test_time_inference']['modality'] == 'scalp_eeg_only', plan
assert plan['test_time_inference']['allow_ieeg'] is False, plan
assert plan['test_time_inference']['allow_beamforming_bridge'] is False, plan
assert plan['scientific_boundary']['feature_matrix_materialized'] is False, plan
assert plan['scientific_boundary']['runtime_comparator_logs_audited'] is False, plan
assert plan['scientific_boundary']['model_training_run'] is False, plan
assert plan['scientific_boundary']['comparator_execution_run'] is False, plan
assert plan['scientific_boundary']['efficacy_metrics_computed'] is False, plan
assert summary['claim_closed'] is True, summary
assert summary['feature_matrix_materialized'] is False, summary
assert summary['runtime_comparator_logs_audited'] is False, summary
assert summary['model_training_run'] is False, summary
assert summary['efficacy_metrics_computed'] is False, summary
print('V5.6 Tranche 2.3 leakage-audit plan passed integrity assertions.')

## 9. Decision gate closeout

In [ ]:
closeout = {
    'tranche2_3_status': 'feature_matrix_leakage_audit_plan_recorded',
    'run_dir': str(run_dir),
    'claim_closed': True,
    'claim_ready': False,
    'feature_matrix_materialized': False,
    'runtime_comparator_logs_audited': False,
    'model_training_run': False,
    'comparator_execution_run': False,
    'efficacy_metrics_computed': False,
    'next_step': 'manual_review_then_implement_feature_matrix_materializer_only_if_leakage_plan_passes',
    'not_allowed_next': [
        'do_not_train_RIFT_Net_Lite_yet',
        'do_not_run_A3_A4_comparators_yet',
        'do_not_open_efficacy_claim',
    ],
}
print(json.dumps(closeout, indent=2))
print('\n=== report preview ===')
print((run_dir / 'v56_feature_matrix_leakage_audit_plan_report.md').read_text()[:2500])